In [2]:
import pandas as pd

reviews = pd.read_csv("../data/raw/reviews.csv.gz")
print(reviews.shape)
print(reviews.columns.tolist())

scored = pd.read_csv("../data/processed/listings_scored.csv", dtype={"id": "int64"})

# κρατάμε μόνο κριτικές για αγγελίες που έμειναν μετά τον καθαρισμό
reviews = reviews[reviews["listing_id"].isin(scored["id"])].copy()
print(len(reviews), "κριτικές για", reviews["listing_id"].nunique(), "αγγελίες")

per_listing = reviews.groupby("listing_id").size()
print(per_listing.describe())

reviews["n_chars"] = reviews["comments"].fillna("").str.len()
print(reviews["n_chars"].describe())
print(reviews["comments"].dropna().sample(5, random_state=1).tolist())

(986851, 6)
['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments']
943161 κριτικές για 12169 αγγελίες
count    12169.000000
mean        77.505218
std        116.379929
min          1.000000
25%          7.000000
50%         30.000000
75%         96.000000
max       1124.000000
dtype: float64
count    943161.000000
mean        242.485945
std         223.023206
min           0.000000
25%          86.000000
50%         181.000000
75%         328.000000
max        5847.000000
Name: n_chars, dtype: float64
['This is an excellent place for the price. Its in a excellent location in Koukaki which is very safe (we were 2 women) and you can walk everywhere. Only 2 blocks away are 2 blocks of restaurants/bars which is where we went everynight. Everything is new and many imenities incl a washer which we took full advantage of the week we were there. When we go back we most definitely will be staying with Vesna again! They even had a delicious bottle of wine waiting for us. Oh yea

In [3]:
from visitath.rag.reviews import clean_reviews

raw_reviews = pd.read_csv("../data/raw/reviews.csv.gz")
clean = clean_reviews(raw_reviews, scored["id"])
print(clean.shape, clean["listing_id"].nunique())
print(clean["comments"].str.len().describe())

(236625, 4) 12066
count    236625.000000
mean        254.853515
std         214.001410
min          30.000000
25%         100.000000
50%         190.000000
75%         339.000000
max        3611.000000
Name: comments, dtype: float64


In [4]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def lang(t):
    try:
        return detect(t)
    except Exception:
        return "unk"

sample = clean.sample(3000, random_state=1).copy()
sample["lang"] = sample["comments"].apply(lang)
print(sample["lang"].value_counts(normalize=True).head(8).round(3))

lang
en    0.617
fr    0.098
el    0.092
de    0.049
es    0.042
it    0.019
tr    0.018
nl    0.009
Name: proportion, dtype: float64


In [6]:
import pandas as pd

df = pd.read_csv("../data/processed/reviews_clean.csv", dtype={"review_id": "int64", "listing_id": "int64"})
dups = df[df.duplicated("review_id", keep=False)].sort_values("review_id")
print(len(df), df["review_id"].nunique(), len(dups))
print(dups.head(6)[["review_id", "listing_id", "date", "comments"]])

146725 146722 6
                 review_id           listing_id        date  \
21452  1739695142969861371  1715130004271272211  2026-07-28   
21432  1739695142969861371  1715130306189383532  2026-07-28   
18817  1744795049734782309  1715130004271272211  2026-08-04   
18728  1744795049734782309  1715130306189383532  2026-08-04   
8491   1765777548760172513  1694004710008126712  2026-09-02   
8547   1765777548760172513  1693481333921218769  2026-09-02   

                                                comments  
21452  One of the best hotels I’ve stayed at. Right a...  
21432  One of the best hotels I’ve stayed at. Right a...  
18817  A value-for-money, fresh-looking hotel in cent...  
18728  A value-for-money, fresh-looking hotel in cent...  
8491   Good value for money in a modern hotel, of cou...  
8547   Good value for money in a modern hotel, of cou...  
